In [67]:
#load the dataset
import pandas as pd
data = pd.read_csv('/home/shreyas/code/HazardX/data/IHMStefanini_industrial_safety_and_health_database.csv')
data.head(5)

,Date,Countries,Local,Industry Sector,Accident Level,Potential Accident Level,Gender,Employee or Third Party,Critical Risk
0,2016-01-01 00:00:00,Country_01,Local_01,Mining,I,IV,Male,Third Party,Pressed
1,2016-01-02 00:00:00,Country_02,Local_02,Mining,I,IV,Male,Employee,Pressurized Systems
2,2016-01-06 00:00:00,Country_01,Local_03,Mining,I,III,Male,Third Party (Remote),Manual Tools
3,2016-01-08 00:00:00,Country_01,Local_04,Mining,I,I,Male,Third Party,Others
4,2016-01-10 00:00:00,Country_01,Local_04,Mining,IV,IV,Male,Third Party,Others


In [68]:
#analyse the dataset
data.info()
data.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439 entries, 0 to 438
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Date                      439 non-null    object
 1   Countries                 439 non-null    object
 2   Local                     439 non-null    object
 3   Industry Sector           439 non-null    object
 4   Accident Level            439 non-null    object
 5   Potential Accident Level  439 non-null    object
 6   Gender                    439 non-null    object
 7   Employee or Third Party   439 non-null    object
 8   Critical Risk             439 non-null    object
dtypes: object(9)
memory usage: 31.0+ KB


,Date,Countries,Local,Industry Sector,Accident Level,Potential Accident Level,Gender,Employee or Third Party,Critical Risk
count,439,439,439,439,439,439,439,439,439
unique,287,3,12,3,5,6,2,3,34
top,2016-02-26 00:00:00,Country_01,Local_03,Mining,I,IV,Male,Third Party,Others
freq,13,263,90,241,328,155,417,189,232


In [69]:
data['Accident Level'].value_counts()
data["Accident Level"].isnull().sum()
data["Accident Level"].unique()

array(['I', 'IV', 'III', 'II', 'V'], dtype=object)

In [70]:
#feature selection
data = data[[
    "Industry Sector",
    "Employee or Third Party",
    "Critical Risk",
    "Accident Level"
]]

In [71]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# encode target only
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(data["Accident Level"])

cat_features = ["Industry Sector", "Employee or Third Party", "Critical Risk"]
preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)],
    remainder="drop",
    sparse_threshold=0.0
)

In [72]:
from sklearn.model_selection import train_test_split

X = data.drop("Accident Level", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

rf_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=1,
    min_samples_split=2,
    max_depth=None,
 )

lr_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1,
 )

models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
}

ordinal_preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            cat_features,
        )
    ],
    remainder="drop",
)

ros = RandomOverSampler(random_state=42)

rf_ros_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
 )

ros_clf = ImbPipeline([
    ("preprocess", ordinal_preprocess),
    ("ros", ros),
    ("model", rf_ros_model),
])

In [74]:
from sklearn.metrics import classification_report, balanced_accuracy_score

def evaluate(name, clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f"\n{name}")
    print(classification_report(y_test, y_pred, digits=3))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
    return y_pred

for name, model in models.items():
    clf = Pipeline([("preprocess", preprocess), ("model", model)])
    y_pred = evaluate(name, clf)

y_pred = evaluate("Random Forest + RandomOverSampler", ros_clf)

/home/shreyas/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/shreyas/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/shreyas/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/shreyas/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being


Logistic Regression
              precision    recall  f1-score   support

           0      0.762     0.242     0.368        66
           1      0.182     0.250     0.211         8
           2      0.143     0.667     0.235         6
           3      0.000     0.000     0.000         6
           4      0.036     0.500     0.067         2

    accuracy                          0.261        88
   macro avg      0.224     0.332     0.176        88
weighted avg      0.599     0.261     0.313        88

Balanced accuracy: 0.33181818181818185

Random Forest
              precision    recall  f1-score   support

           0      0.711     0.485     0.577        66
           1      0.000     0.000     0.000         8
           2      0.125     0.333     0.182         6
           3      0.000     0.000     0.000         6
           4      0.000     0.000     0.000         2

    accuracy                          0.386        88
   macro avg      0.167     0.164     0.152        88
we

In [75]:
import numpy as np
from sklearn.metrics import recall_score

labels = list(target_encoder.classes_)
per_class_recall = recall_score(y_test, y_pred, average=None, labels=np.arange(len(labels)))

print("Macro recall:", recall_score(y_test, y_pred, average="macro"))
print("Weighted recall:", recall_score(y_test, y_pred, average="weighted"))
display(pd.DataFrame({"class": labels, "recall": per_class_recall}).sort_values("recall"))

test_dist = pd.Series(y_test).map(lambda i: labels[i]).value_counts(normalize=True)
print("\nTest class distribution:")
display(test_dist)

Macro recall: 0.1696969696969697
Weighted recall: 0.29545454545454547


,class,recall
1,II,0.000000
4,V,0.000000
3,IV,0.166667
2,III,0.333333
0,I,0.348485



Test class distribution:


I      0.750000
II     0.090909
III    0.068182
IV     0.068182
V      0.022727
Name: proportion, dtype: float64